In [0]:
import logging
import pyspark.sql.functions as F
from pyspark.sql.window import Window 

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

In [0]:
catalog_name = "supply_chain_daily_demand"
bronze_schema_name = "bronze"
bronze_table_name = "raw_daily_demand"
silver_schema_name = "silver"
silver_table_name = "transformed_daily_demand"

In [0]:
class Silver:
    def load_from_bronze(self):
        bronze_table_path = f"{catalog_name}.{bronze_schema_name}.{bronze_table_name}"

        logging.info(f"Loading the delta table from path: {bronze_table_path}.")
        self.data = spark.read.table(bronze_table_path)
        logging.info(f"Bronze delta table is loaded.")

    def remove_duplicates(self):
        window_config = Window.partitionBy("transaction_id").orderBy("transaction_date")

        count_before_removing = self.data.count()

        logging.info(f"Duplication removing started, total record count: {count_before_removing}")
        self.data = self.data.withColumn(
            "duplicate_count",
            F.row_number().over(window_config)
        ).filter(
            F.col("duplicate_count") == 1
        ).drop("duplicate_count")

        count_after_removing = self.data.count()
        logging.info(f"Duplicates are removed: {count_before_removing - count_after_removing}")

    def write_to_silver(self):
        logging.info("Adding the silver_ingestion_timestamp column.")
        self.data = self.data.withColumn("silver_ingestion_timestamp", F.current_timestamp())

        silver_table_path = f"{catalog_name}.{silver_schema_name}.{silver_table_name}"
        logging.info(f"Writing the dataframe to the delta table. Path: {silver_table_path}")

        self.data.write.format("delta")\
                       .mode("overwrite")\
                       .saveAsTable(silver_table_path)

        logging.info("Table saved.")

In [0]:
silver = Silver()
silver.load_from_bronze()
silver.remove_duplicates()
silver.write_to_silver()

In [0]:
#spark.table("supply_chain_daily_demand.silver.transformed_daily_demand").show()